# Degrees of fully navigable graphs: effective-dimension sweep

For each target effective dimension in `D_EFF_RATIOS` ($d_\mathrm{eff}/d$, where $d_\mathrm{eff}$ is the participation ratio), `get_alpha` finds the power-law exponent $\alpha$ whose covariance eigenvalues $k^{-\alpha}$ ($k = 1..d$) give that ratio. `generate_gaussian` then draws `N` points, and the robust-prune navigable graph is built with `batchedEuclideanRobustPrune` from `utils.py` (`PRUNE_ALPHA = 1`: the basic coverage rule). Only vertex degrees are kept.

- **Out-degree** of a vertex = size of its pruned neighborhood. Each neighborhood depends only on the dataset, so a random sample of sources (`SOURCES = k`) gives an unbiased picture of the out-degree distribution.
- **In-degree** is counted as batches finish; it is complete only once every vertex is processed (`SOURCES = None`).

Neither the graphs nor the datasets are stored. Each dataset is regenerated from `SEED` (same seed for every ratio, so only the covariance scaling changes), and each ratio's degrees are checkpointed to `results/navigable_degrees/<tag>-degrees.npz` every `CHECKPOINT_EVERY` batches. Rerunning resumes from the checkpoints, processing sources in the same shuffled order.

In [ ]:
import os
import sys
import types
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from tqdm.auto import tqdm

from paths import RESULTS_DIR

D_EFF_RATIOS = []          # target d_eff / d values to sweep, each in [1/D, 1], e.g. [0.1, 0.25, 0.5, 0.75, 1.0]
N = 1_000_000
D = 128
SEED = 42
PRUNE_ALPHA = 1.0       # alpha-reachability; 1.0 = basic robust prune (fully navigable graph)
USE_GPU = None          # None: use CuPy if it imports and sees a CUDA GPU (HPC), else NumPy (e.g. a MacBook)
BATCH_SIZE = None       # None: 512 on GPU, 64 on CPU. Sources pruned together; memory ~ 5 * BATCH_SIZE * N * 4 bytes (~10 GB at 512, N = 1M)
SOURCES = None          # None: every vertex (needed for in-degrees); int k: a random sample of k sources
CHECKPOINT_EVERY = 10   # batches between checkpoint writes
OUT_DIR = RESULTS_DIR / "navigable_degrees"

assert D_EFF_RATIOS, "set D_EFF_RATIOS to the d_eff / d values to sweep"


def cuda_available():
    """True if the real CuPy imports and sees at least one CUDA device."""
    try:
        import cupy
        return not getattr(cupy, "_placeholder", False) and cupy.cuda.runtime.getDeviceCount() > 0
    except Exception:
        return False


if USE_GPU is None:
    USE_GPU = cuda_available()
if BATCH_SIZE is None:
    BATCH_SIZE = 512 if USE_GPU else 64
print(f"USE_GPU = {USE_GPU}, BATCH_SIZE = {BATCH_SIZE}")


def tag(ratio):
    return f"gaussian-n{N}-d{D}-deff{ratio:g}-seed{SEED}-alpha{PRUNE_ALPHA:g}".replace(".", "p")

## Datasets

In [ ]:
def generate_gaussian(n=1_000_000, d=128, alpha=0.68, seed=42):
    rng = np.random.default_rng(seed)

    # Power-law eigenvalues of covariance matrix
    eigenvalues = np.arange(1, d + 1, dtype=np.float64) ** (-alpha)

    # Effective dimension (participation ratio)
    d_eff = eigenvalues.sum()**2 / np.sum(eigenvalues**2)
    print(f"Effective dimension: {d_eff/d}")

    # Generate Gaussian points with diagonal covariance
    X = rng.standard_normal((n, d), dtype=np.float32)
    X *= np.sqrt(eigenvalues).astype(np.float32)

    return X, eigenvalues


def get_alpha(target_ratio, d=128):
    """Find alpha such that d_eff / d = target_ratio."""
    if target_ratio == 1:
        return 0.0

    if target_ratio < 1 / d:
        raise ValueError(f"target_ratio must be >= 1/d = {1/d:.6f}.")

    def ratio(alpha):
        eigs = np.arange(1, d + 1, dtype=float) ** (-alpha)
        return eigs.sum()**2 / (d * np.sum(eigs**2))

    return brentq(lambda a: ratio(a) - target_ratio, 0, 100)


ALPHAS = {r: get_alpha(r, d=D) for r in D_EFF_RATIOS}
pd.DataFrame({"d_eff / d": list(ALPHAS), "power-law alpha": list(ALPHAS.values()),
              "file": [f"{tag(r)}-degrees.npz" for r in ALPHAS]})

## Robust prune

`USE_GPU` picks `batchedEuclideanRobustPrune` (CuPy) or `batchedEuclideanRobustPruneCPU` (NumPy / BLAS). `USE_GPU = None` checks that CuPy really imports and sees a CUDA device. `utils.py` imports CuPy and numba at the top; on the CPU path, empty placeholder modules (marked `_placeholder`) are registered for whichever of them fails to import, so the NumPy functions can be imported. If a kernel that took the CPU path is later switched to the GPU path, the next cell stops with a request to restart the kernel. `utils.py` itself is unchanged, and installed packages are never replaced.

In [ ]:
if USE_GPU:
    import cupy as cp
    if getattr(cp, "_placeholder", False):
        raise RuntimeError("this kernel holds the CPU-path placeholder for cupy from an earlier run: "
                           "restart the kernel, then run all cells")
    from utils import batchedEuclideanRobustPrune as prune, precomputeAugMatrix as augment
else:
    # Placeholders (marked _placeholder) only for what fails to import, just so `import utils` succeeds;
    # the CPU functions don't use them.
    try:
        import cupy, cupyx.scipy.spatial.distance
    except Exception:
        for name in ("cupy", "cupyx", "cupyx.scipy", "cupyx.scipy.spatial", "cupyx.scipy.spatial.distance"):
            sys.modules[name] = types.ModuleType(name)
            sys.modules[name]._placeholder = True
        sys.modules["cupyx.scipy.spatial.distance"].cdist = None
    try:
        import numba
    except ImportError:
        numba = sys.modules["numba"] = types.ModuleType("numba")
        numba.njit, numba.prange, numba._placeholder = (lambda *a, **k: (lambda f: f)), range, True
    from utils import batchedEuclideanRobustPruneCPU as prune, precomputeAugMatrixCPU as augment
print(f"prune: {prune.__name__}  (batch size {BATCH_SIZE})")


def load_state(ckpt):
    """(out_degree, in_degree, done) from a checkpoint, or fresh arrays (out_degree -1 = not computed)."""
    if ckpt.exists():
        z = np.load(ckpt)
        assert int(z["n"]) == N, f"{ckpt} is for n={int(z['n'])}, not {N}"
        print(f"  resuming: {int(z['done'].sum()):,} of {N:,} sources already computed")
        return z["out_degree"], z["in_degree"], z["done"]
    return np.full(N, -1, dtype=np.int32), np.zeros(N, dtype=np.int64), np.zeros(N, dtype=bool)


def save_state(ckpt, ratio, out_degree, in_degree, done):
    """Write a checkpoint atomically (a crash mid-write keeps the previous one)."""
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    tmp = ckpt.with_name(ckpt.stem + ".tmp.npz")
    np.savez(tmp, out_degree=out_degree, in_degree=in_degree, done=done, n=N, d=D, d_eff_ratio=ratio,
             spectrum_alpha=ALPHAS[ratio], seed=SEED, prune_alpha=PRUNE_ALPHA)
    os.replace(tmp, ckpt)


def run_ratio(ratio):
    """Build (or resume) the degrees for one d_eff / d value; returns (out_degree, in_degree, done)."""
    ckpt = OUT_DIR / f"{tag(ratio)}-degrees.npz"
    print(f"d_eff/d = {ratio:g}  (alpha = {ALPHAS[ratio]:.4f})  -> {ckpt}")
    out_degree, in_degree, done = load_state(ckpt)

    # Fixed shuffled order (seeded), so a resumed run continues the same sample of sources.
    order = np.random.default_rng(SEED + 1).permutation(N)
    if SOURCES is not None:
        order = order[:SOURCES]
    todo = order[~done[order]]
    if len(todo) == 0:
        print("  already complete")
        return out_degree, in_degree, done

    X, _ = generate_gaussian(N, D, ALPHAS[ratio], SEED)
    data = cp.asarray(X) if USE_GPU else X
    X_aug = augment(data)
    for b, start in enumerate(tqdm(range(0, len(todo), BATCH_SIZE), unit="batch", desc=f"d_eff/d={ratio:g}")):
        batch = todo[start : start + BATCH_SIZE].tolist()
        neighborhoods = prune(batch, data, X_aug, alpha=PRUNE_ALPHA)
        for source, hood in zip(batch, neighborhoods):
            nbrs = np.fromiter((v for v, _ in hood), dtype=np.int64, count=len(hood))
            out_degree[source] = len(nbrs)
            in_degree[nbrs] += 1        # a source's neighbors are distinct
            done[source] = True
        if (b + 1) % CHECKPOINT_EVERY == 0:
            save_state(ckpt, ratio, out_degree, in_degree, done)
    save_state(ckpt, ratio, out_degree, in_degree, done)

    del data, X_aug
    if USE_GPU:
        cp.get_default_memory_pool().free_all_blocks()   # release this ratio's arrays before the next
    print(f"  {int(done.sum()):,} of {N:,} sources computed")
    return out_degree, in_degree, done

## Sweep

In [ ]:
results = {ratio: run_ratio(ratio) for ratio in D_EFF_RATIOS}

## Degree statistics

Out-degrees over the computed sources; in-degrees only for ratios where every vertex is done.

In [ ]:
def summary(x):
    p = np.percentile(x, [1, 10, 50, 90, 99])
    return {"count": len(x), "mean": x.mean(), "std": x.std(), "min": x.min(),
            "p1": p[0], "p10": p[1], "median": p[2], "p90": p[3], "p99": p[4], "max": x.max()}

rows = []
for ratio, (out_degree, in_degree, done) in results.items():
    rows.append({"d_eff / d": ratio, "alpha": ALPHAS[ratio], "degree": "out", **summary(out_degree[done])})
    if done.all():
        rows.append({"d_eff / d": ratio, "alpha": ALPHAS[ratio], "degree": "in", **summary(in_degree)})
stats = pd.DataFrame(rows).set_index(["d_eff / d", "degree"])
stats.style.format("{:,.1f}").format("{:,.0f}", subset=["count", "min", "max"]).format("{:.4f}", subset=["alpha"])

## Plot

Left: out-degree against d_eff / d (median, mean, and 99th percentile). Right: the out-degree distribution for each ratio, on shared log-spaced bins, as a fraction of computed sources.

In [ ]:
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]   # fixed categorical order
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
})

out = stats.xs("out", level="degree")
fig, (ax_s, ax_h) = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
for col, color, label in (("median", SERIES[0], "median"), ("mean", SERIES[1], "mean"), ("p99", SERIES[2], "99th pct")):
    ax_s.plot(out.index, out[col], color=color, lw=2, marker="o", ms=8, mec=SURFACE, label=label)
ax_s.set(xlabel="d_eff / d (effective dimension / d)", ylabel="out-degree", title="Out-degree vs effective dimension")
ax_s.legend(frameon=False)

degs = {r: results[r][0][results[r][2]] for r in D_EFF_RATIOS}
edges = np.geomspace(max(1, min(x.min() for x in degs.values())), max(x.max() for x in degs.values()) + 1, 50)
if len(degs) > len(SERIES):
    print(f"showing the first {len(SERIES)} ratios in the histogram")
for color, (ratio, x) in zip(SERIES, degs.items()):
    counts, _ = np.histogram(x, bins=edges)
    ax_h.stairs(counts / len(x), edges, color=color, lw=2, label=f"d_eff/d = {ratio:g}")
ax_h.set(xscale="log", xlabel="out-degree (log scale)", ylabel="fraction of sources", title="Out-degree distribution")
ax_h.legend(frameon=False)
fig.suptitle(f"Robust-prune navigable graphs: n={N:,}, d={D}", fontweight="bold", color=INK)
plt.show()

## Results: degree vs effective dimension

One row per dataset in the sweep, in the layout of `effective_dim.ipynb`: the covariance's effective dimension (d_eff, d_eff / d, and how many top directions hold 50 / 90 / 99% of the variance), next to the graph's degree statistics. The spectrum columns use the exact covariance eigenvalues $k^{-\alpha}$; with `N` points the sample covariance agrees closely. In-degree columns are filled only for ratios where every vertex was processed.

In [ ]:
def dims_for(eigs, q):
    """Fewest top directions holding fraction q of the variance (eigenvalues largest first)."""
    return int(np.searchsorted(np.cumsum(eigs) / eigs.sum(), q) + 1)

table = []
for ratio, (out_degree, in_degree, done) in results.items():
    eigs = np.sort(np.arange(1, D + 1, dtype=np.float64) ** (-ALPHAS[ratio]))[::-1]
    d_eff = eigs.sum() ** 2 / (eigs**2).sum()
    out = out_degree[done]
    table.append({
        "d_eff / d": d_eff / D, "d_eff": d_eff, "alpha": ALPHAS[ratio],
        "dims for 50%": dims_for(eigs, 0.5), "dims for 90%": dims_for(eigs, 0.9), "dims for 99%": dims_for(eigs, 0.99),
        "sources": int(done.sum()),
        "out mean": out.mean(), "out median": np.median(out), "out p99": np.percentile(out, 99), "out max": out.max(),
        "in p99": np.percentile(in_degree, 99) if done.all() else np.nan,
        "in max": in_degree.max() if done.all() else np.nan,
    })
table = pd.DataFrame(table)
table.style.format({"d_eff / d": "{:.3f}", "d_eff": "{:.1f}", "alpha": "{:.4f}", "sources": "{:,}",
                    "out mean": "{:.1f}", "out median": "{:.0f}", "out p99": "{:.0f}", "out max": "{:.0f}",
                    "in p99": "{:.0f}", "in max": "{:.0f}"}, na_rep="–").hide(axis="index")

In [ ]:
# table.to_csv(OUT_DIR / (f"gaussian-n{N}-d{D}-seed{SEED}-alpha{PRUNE_ALPHA:g}-sweep".replace(".", "p") + ".csv"), index=False)